# AIT201 Group 4：Used Car Price Prediction
More details can be found on: https://github.com/Stanley-oss/CarPricePred  
We designed **an interactive interface** for users to input the real-world data for prediction!  
Welcome to have a try!

## Part 1: Imports and Configuration

In [1]:
import math
import os
import random
import re
import warnings
from dataclasses import dataclass
from datetime import datetime
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.utils.data as data
from catboost import CatBoostRegressor, Pool
from sklearn.ensemble import GradientBoostingRegressor, IsolationForest
from sklearn.impute import KNNImputer
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings('ignore')

CSV_PATH = "Full_dataset.csv"
INR2USD = 83.4
CATBOOST_ITER = 5000
RESNET_EPOCHS = 100
RANDOM_STATE = 42

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(RANDOM_STATE)

## Part 2: Preprocessing and Outlier Treatment

In [2]:
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
else:
    data_mock = {
        'Brand': ['Toyota'] * 100,
        'Model': ['Corolla'] * 100,
        'Age': np.random.randint(1, 10, 100),
        'Kilometer': np.random.randint(10000, 100000, 100),
        'Fuel Type': ['Petrol'] * 100,
        'Transmission': ['Automatic'] * 100,
        'Engine': [1800] * 100,
        'Max Power': [140] * 100,
        'Seats': [5] * 100,
        'Price': np.random.randint(50000, 100000, 100),
        'Year': [2019] * 100,
        'listing_date': ['2023-01-01'] * 100
    }
    df = pd.DataFrame(data_mock)

cols_to_keep = [
    'Brand', 'Model', 'Age', 'Kilometer', 'Fuel Type', 
    'Transmission', 'Engine', 'Max Power', 'Seats', 'Price', 'Year', 'listing_date'
]
cols_to_keep = [c for c in cols_to_keep if c in df.columns]
df = df[cols_to_keep].copy()

if 'Brand' in df.columns:
    df['Brand'] = df['Brand'].fillna('Unknown')
if 'Model' in df.columns:
    df['Model'] = df['Model'].fillna('Unknown')

if df['Max Power'].dtype == 'O':
    df['Max Power'] = pd.to_numeric(
        df['Max Power'].astype(str).str.extract(r'(\d+\.?\d*)')[0], 
        errors='coerce'
    )

df['Log_Price'] = np.log1p(df['Price'])

cat_cols = ['Fuel Type', 'Transmission']
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')
        df[col + '_Code'] = LabelEncoder().fit_transform(df[col].astype(str))

cols_num = ['Engine', 'Max Power', 'Seats']
cols_num = [c for c in cols_num if c in df.columns]

imputer = KNNImputer(n_neighbors=5)
df[cols_num] = imputer.fit_transform(df[cols_num])

df['Age_Squared'] = df['Age'] ** 2
df['Km_per_Year'] = df['Kilometer'] / (df['Age'] + 0.1)
df['Power_per_Seat'] = df['Max Power'] / (df['Seats'] + 1e-5)

base_features = ['Age', 'Kilometer', 'Engine', 'Max Power', 'Seats', 'Fuel Type_Code', 'Transmission_Code']
fe_features = base_features + ['Age_Squared', 'Km_per_Year', 'Power_per_Seat']
fe_features = [f for f in fe_features if f in df.columns]
final_features = fe_features 

df_fe = df.copy()

iso_feats = [f for f in ['Age', 'Kilometer', 'Engine', 'Max Power'] if f in df.columns]
iso = IsolationForest(contamination=0.01, random_state=RANDOM_STATE)
iso_labels = iso.fit_predict(df_fe[iso_feats])
mask_layer1 = (iso_labels == 1)

X_clean = df_fe[final_features]
y_clean = df_fe['Log_Price']

gbr_low = GradientBoostingRegressor(loss='quantile', alpha=0.01, n_estimators=50, max_depth=5, random_state=RANDOM_STATE)
gbr_high = GradientBoostingRegressor(loss='quantile', alpha=0.99, n_estimators=50, max_depth=5, random_state=RANDOM_STATE)
gbr_low.fit(X_clean, y_clean)
gbr_high.fit(X_clean, y_clean)

mask_layer2 = (y_clean >= gbr_low.predict(X_clean)) & (y_clean <= gbr_high.predict(X_clean))
mask_both = mask_layer1 & mask_layer2

df_final = df_fe[mask_both].copy()
print(f"Original size: {len(df)}, Cleaned size: {len(df_final)}")

Original size: 12646, Cleaned size: 12360


## Part 3: CatBoost Model

In [3]:
TARGET_COL = "Price"
DATE_COL = "listing_date"
BRAND_COL = "Brand"
MODEL_COL = "Model"
YEAR_COL = "Year"
AGE_COL = "Age"
MILE_COL = "Kilometer"
HP_COL = "Max Power"
ENGINE_COL = "Engine"
GEAR_COL = "Transmission"
FUEL_COL = "Fuel Type"
SEATS_COL = "Seats"
ALPHA = 0.2

CATBOOST_PARAMS = dict(
    depth=8,
    learning_rate=0.035,
    l2_leaf_reg=6.0,
    loss_function="Quantile:alpha={alpha}",
    iterations=CATBOOST_ITER,
    random_seed=RANDOM_STATE,
    border_count=254,
    verbose=False,
    thread_count=-1,
    od_type="Iter",
    od_wait=300,
    subsample=0.9,
    rsm=0.9,
    bootstrap_type="Bernoulli",
)


def to_number(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x)
    m = re.search(r"[-+]?\d*\.?\d+", s.replace(",", ""))
    return float(m.group(0)) if m else np.nan


def winsorize_series(s, lower=0.005, upper=0.995):
    s = pd.to_numeric(s, errors="coerce")
    lo, hi = (s.quantile(lower), s.quantile(upper))
    return (s.clip(lo, hi), lo, hi)


def safe_log1p(x):
    x = np.maximum(x, 0)
    return np.log1p(x)


def standardize_enum(series, mapping, default="Unknown"):
    s = series.astype(str).str.strip().str.lower()
    return s.map(mapping).fillna(default)


def make_age(df):
    if AGE_COL in df.columns:
        age = pd.to_numeric(df[AGE_COL], errors="coerce")
    elif YEAR_COL in df.columns:
        current_year = datetime.now().year
        age = current_year - pd.to_numeric(df[YEAR_COL], errors="coerce")
    else:
        age = pd.Series(np.nan, index=df.index)
    return age.where(age >= 0, np.nan)


def age_bin(a):
    if pd.isna(a):
        return "age:Unknown"
    a = float(a)
    if a < 3:
        return "age:0-3"
    if a < 8:
        return "age:3-8"
    return "age:8+"


def get_period_series(df):
    if DATE_COL in df.columns:
        d = pd.to_datetime(df[DATE_COL], errors="coerce")
        p = d.dt.year.astype("Int64").astype(str)
    elif YEAR_COL in df.columns:
        p = pd.to_numeric(df[YEAR_COL], errors="coerce").astype("Int64").astype(str)
    else:
        p = pd.Series(["Unknown"] * len(df))
    return p.fillna("Unknown")


def make_period_bin_from_series(period_series):
    yy = pd.to_numeric(period_series, errors="coerce")
    out = []
    for v in yy:
        if np.isnan(v):
            out.append("Unknown")
        else:
            lo = int(v) // 2 * 2
            out.append(f"{lo}-{lo + 1}")
    return pd.Series(out, index=period_series.index)


def compute_residual_by_period(y_log_true, y_log_pred, period_series, agg="median"):
    r = y_log_true - y_log_pred
    df_r = pd.DataFrame({"r": r, "period": period_series})
    if agg == "median":
        R = df_r.groupby("period")["r"].median()
    else:
        R = df_r.groupby("period")["r"].mean()
    R = R - R.mean()
    return R.to_dict()


def fit_time_dummy(R_dict):
    gamma = dict(R_dict)
    M = {k: float(np.exp(v)) for k, v in gamma.items()}
    return gamma, M


def smooth_market_index(gamma_dict, alpha=0.25):
    keys = sorted([k for k in gamma_dict.keys() if k.isdigit()])
    sm = {}
    last = 0.0
    for k in keys:
        g = float(gamma_dict[k])
        last = alpha * g + (1 - alpha) * last
        sm[k] = last
    return {k: float(np.exp(v)) for k, v in sm.items()}


def build_feature_df(df_raw):
    df = df_raw.copy()
    for col in [TARGET_COL, MILE_COL, HP_COL, ENGINE_COL, YEAR_COL, AGE_COL, SEATS_COL]:
        if col in df.columns:
            df[col] = df[col].apply(to_number)

    if GEAR_COL in df.columns:
        df[GEAR_COL] = standardize_enum(
            df[GEAR_COL],
            {
                "a": "Automatic",
                "auto": "Automatic",
                "automatic": "Automatic",
                "m": "Manual",
                "man": "Manual",
                "manual": "Manual",
            },
            default="Unknown",
        )

    if FUEL_COL in df.columns:
        df[FUEL_COL] = standardize_enum(
            df[FUEL_COL],
            {
                "petrol": "Petrol",
                "gasoline": "Petrol",
                "diesel": "Diesel",
                "cng": "Other",
                "lpg": "Other",
                "hybrid": "Other",
                "electric": "Other",
                "other": "Other",
            },
            default="Other",
        )

    for col in [TARGET_COL, MILE_COL, HP_COL, ENGINE_COL]:
        if col in df.columns:
            df[col], _, _ = winsorize_series(df[col], 0.005, 0.995)

    df["car_age"] = make_age(df)

    if MILE_COL in df.columns:
        df["log1p_mileage"] = safe_log1p(df[MILE_COL])

    if MILE_COL in df.columns and "car_age" in df.columns:
        age_eps = df["car_age"].replace(0, 0.25)
        df["avg_km_per_year"] = df[MILE_COL] / age_eps

    if HP_COL in df.columns and GEAR_COL in df.columns:
        is_auto = (df[GEAR_COL] == "Automatic").astype(int)
        df["hp_x_auto"] = df[HP_COL] * is_auto

    if HP_COL in df.columns and "avg_km_per_year" in df.columns:
        with np.errstate(divide="ignore", invalid="ignore"):
            df["hp_div_avgkm"] = df[HP_COL] / np.where(df["avg_km_per_year"] > 0, df["avg_km_per_year"], np.nan)

    if HP_COL in df.columns and ENGINE_COL in df.columns:
        with np.errstate(divide="ignore", invalid="ignore"):
            df["power_per_cc"] = df[HP_COL] / np.where(df[ENGINE_COL] > 0, df[ENGINE_COL], np.nan)

    if ENGINE_COL in df.columns and SEATS_COL in df.columns:
        with np.errstate(divide="ignore", invalid="ignore"):
            df["cc_per_seat"] = df[ENGINE_COL] / np.where(df[SEATS_COL] > 0, df[SEATS_COL], np.nan)

    if BRAND_COL in df.columns:
        cnt_brand = df[BRAND_COL].astype(str).map(df[BRAND_COL].astype(str).value_counts())
        df["brand_count"] = cnt_brand

    if MODEL_COL in df.columns:
        cnt_model = df[MODEL_COL].astype(str).map(df[MODEL_COL].astype(str).value_counts())
        df["model_count"] = cnt_model

    if BRAND_COL in df.columns and MODEL_COL in df.columns:
        bm = df[BRAND_COL].astype(str) + "§" + df[MODEL_COL].astype(str)
        cnt_bm = bm.map(bm.value_counts())
        df["brand_model_count"] = cnt_bm

    numeric_candidates = [
        YEAR_COL,
        AGE_COL,
        MILE_COL,
        HP_COL,
        ENGINE_COL,
        SEATS_COL,
        "car_age",
        "log1p_mileage",
        "avg_km_per_year",
        "hp_x_auto",
        "hp_div_avgkm",
        "power_per_cc",
        "cc_per_seat",
        "brand_count",
        "model_count",
        "brand_model_count",
    ]
    numeric_candidates = [c for c in numeric_candidates if c in df.columns]

    categorical_candidates = [BRAND_COL, MODEL_COL, GEAR_COL, FUEL_COL]
    categorical_candidates = [c for c in categorical_candidates if c in df.columns]

    y = pd.to_numeric(df[TARGET_COL], errors="coerce")
    X = pd.DataFrame(index=df.index)

    for col in numeric_candidates:
        col_num = pd.to_numeric(df[col], errors="coerce")
        miss_flag = col_num.isna().astype(int)
        med = np.nanmedian(col_num)
        X[col] = np.where(np.isnan(col_num), med, col_num)
        X[col + "_missing"] = miss_flag

    for col in categorical_candidates:
        X[col] = df[col].astype(str).replace({"nan": "Unknown", "None": "Unknown"})

    X["period"] = get_period_series(df)
    X["period_bin"] = make_period_bin_from_series(X["period"])
    X["period"] = X["period"].astype(str)
    X["period_bin"] = X["period_bin"].astype(str)

    cat_cols = categorical_candidates.copy()
    for c in ["period", "period_bin"]:
        if c in X.columns and c not in cat_cols:
            cat_cols.append(c)

    keep = ~y.isna()
    return X.loc[keep].reset_index(drop=True), y.loc[keep].reset_index(drop=True), cat_cols


def finite_sample_quantile(scores, q):
    s = np.sort(np.asarray(scores))
    n = len(s)
    if n == 0:
        return 0.0
    rank = int(math.ceil((n + 1) * q)) - 1
    rank = min(max(rank, 0), n - 1)
    return float(s[rank])


def build_group_keys_for_cqr(df_va):
    a = df_va["car_age"].apply(age_bin) if "car_age" in df_va.columns else pd.Series(["age:Unknown"] * len(df_va))
    f = df_va[FUEL_COL] if FUEL_COL in df_va.columns else pd.Series(["fuel:Unknown"] * len(df_va))
    t = df_va[GEAR_COL] if GEAR_COL in df_va.columns else pd.Series(["gear:Unknown"] * len(df_va))
    pb = df_va["period_bin"] if "period_bin" in df_va.columns else pd.Series(["Unknown"] * len(df_va))
    key = a.astype(str).str.cat(f.astype(str), sep="|").str.cat(t.astype(str), sep="|").str.cat(pb.astype(str), sep="|")
    return key


def cqr_asymmetric_ratio_global_and_group(y_true, p50, p10, p90, group_keys, alpha=ALPHA, min_group_size=120):
    y_true = np.asarray(y_true)
    p50 = np.asarray(p50)
    p10 = np.asarray(p10)
    p90 = np.asarray(p90)
    eps = 1e-6
    s_lo_all = np.maximum(0.0, p10 - y_true) / np.maximum(eps, p50)
    s_hi_all = np.maximum(0.0, y_true - p90) / np.maximum(eps, p50)
    q_lo_global = finite_sample_quantile(s_lo_all, 1.0 - alpha)
    q_hi_global = finite_sample_quantile(s_hi_all, 1.0 - alpha)
    q_lo_groups = {}
    q_hi_groups = {}
    g_keys = pd.Series(group_keys).astype(str).values
    uniq = np.unique(g_keys)
    for g in uniq:
        idx = g_keys == g
        n_g = int(idx.sum())
        if n_g >= min_group_size:
            q_lo_groups[g] = finite_sample_quantile(s_lo_all[idx], 1.0 - alpha)
            q_hi_groups[g] = finite_sample_quantile(s_hi_all[idx], 1.0 - alpha)
    return (q_lo_global, q_hi_global), (q_lo_groups, q_hi_groups)


def compute_brand_model_calibration(
    y_true, p50_m, X_va, lam_age=10.0, lam_bm=15.0, lam_brand=20.0, min_cnt_age=3, min_cnt_bm=5, min_cnt_brand=10
):
    if BRAND_COL not in X_va.columns:
        return {}
    brand = X_va[BRAND_COL].astype(str).fillna("Unknown")
    model = X_va[MODEL_COL].astype(str).fillna("Unknown") if MODEL_COL in X_va.columns else pd.Series(["Unknown"] * len(X_va))
    age_bin_series = X_va["car_age"].apply(age_bin) if "car_age" in X_va.columns else pd.Series(["age:Unknown"] * len(X_va))
    y_true = np.asarray(y_true)
    p50_m = np.asarray(p50_m)
    log_r = np.log(np.maximum(y_true, 1e-6)) - np.log(np.maximum(p50_m, 1e-6))
    global_log_med = float(np.median(log_r))
    df_c = pd.DataFrame({"brand": brand.values, "model": model.values, "age_bin": age_bin_series.values, "log_r": log_r})

    def _build_level(group_cols, lam, min_cnt):
        res = {}
        grp = df_c.groupby(group_cols)["log_r"]
        for key, s in grp:
            s = s.dropna()
            n = int(s.size)
            if n < min_cnt:
                continue
            med = float(s.median())
            w = n / (n + lam)
            log_c = (1.0 - w) * global_log_med + w * med
            coef = float(np.exp(log_c))
            if isinstance(key, tuple):
                key_str = "|".join(str(k) for k in key)
            else:
                key_str = str(key)
            res[key_str] = coef
        return res

    return {
        "global_log_median": global_log_med,
        "levels": {
            "brand_model_age": _build_level(["brand", "model", "age_bin"], lam_age, min_cnt_age),
            "brand_model": _build_level(["brand", "model"], lam_bm, min_cnt_bm),
            "brand": _build_level(["brand"], lam_brand, min_cnt_brand),
        },
    }


def get_brand_model_multiplier(brand, model, age_bin_str, cal_cfg):
    if not cal_cfg:
        return 1.0
    levels = cal_cfg.get("levels", {})
    lvl_bma = levels.get("brand_model_age", {})
    lvl_bm = levels.get("brand_model", {})
    lvl_brand = levels.get("brand", {})
    b, m, ab = str(brand), str(model), str(age_bin_str)
    if f"{b}|{m}|{ab}" in lvl_bma:
        return float(lvl_bma[f"{b}|{m}|{ab}"])
    if f"{b}|{m}" in lvl_bm:
        return float(lvl_bm[f"{b}|{m}"])
    if b in lvl_brand:
        return float(lvl_brand[b])
    return 1.0


def compute_global_tail_weights(y_log, n_bins=30, clip_min=0.5, clip_max=3.0):
    y_log = np.asarray(y_log, dtype=float)
    if y_log.size == 0:
        return np.ones(0, dtype=float)
    hist, edges = np.histogram(y_log, bins=n_bins)
    hist = hist.astype(float) + 1e-6
    bin_idx = np.searchsorted(edges, y_log, side="right") - 1
    bin_idx = np.clip(bin_idx, 0, len(hist) - 1)
    freq = hist[bin_idx] / hist.sum()
    w = 1.0 / (freq + 1e-8)
    w /= np.mean(w)
    return np.clip(w, clip_min, clip_max)


def compute_brand_price_weights(y, brand_series, min_brand_samples=40, max_bins=10, clip_min=0.5, clip_max=3.0):
    y = np.asarray(y, dtype=float)
    w = np.ones(y.size, dtype=float)
    brands = pd.Series(brand_series).astype(str).fillna("Unknown").values
    for b in np.unique(brands):
        idx = np.where(brands == b)[0]
        if idx.size < min_brand_samples:
            continue
        y_b = y[idx]
        y_b = y_b[np.isfinite(y_b)]
        if y_b.size < min_brand_samples or np.allclose(y_b.min(), y_b.max()):
            continue
        q25, q75 = np.percentile(y_b, [25, 75])
        iqr = q75 - q25
        if iqr <= 0:
            continue
        h = 2.0 * iqr / (y_b.size ** (1.0 / 3.0))
        if h <= 0:
            continue
        n_bins = int(np.ceil((y_b.max() - y_b.min()) / h))
        n_bins = max(1, min(max_bins, n_bins))
        hist, edges = np.histogram(y_b, bins=n_bins)
        hist = hist.astype(float) + 1e-6
        bin_idx_b = np.searchsorted(edges, y[idx], side="right") - 1
        bin_idx_b = np.clip(bin_idx_b, 0, len(hist) - 1)
        freq_b = hist[bin_idx_b] / hist.sum()
        w_b = 1.0 / (freq_b + 1e-8)
        w_b /= np.mean(w_b)
        w[idx] = np.clip(w_b, clip_min, clip_max)
    return w


def build_sample_weights(y_tr, y_tr_log, brand_tr):
    w_global = compute_global_tail_weights(y_tr_log)
    w_brand = compute_brand_price_weights(y_tr, brand_tr)
    w = w_global * w_brand
    return w / np.mean(w)


def train_quantile_model(X_tr, y_tr, X_va, y_va, alpha, sample_weight=None):
    params = CATBOOST_PARAMS.copy()
    params["loss_function"] = f"Quantile:alpha={alpha}"
    cats = [c for c in X_tr.columns if X_tr[c].dtype == "object"]
    if sample_weight is not None:
        train_pool = Pool(X_tr, y_tr, cat_features=cats, weight=sample_weight)
    else:
        train_pool = Pool(X_tr, y_tr, cat_features=cats)
    val_pool = Pool(X_va, y_va, cat_features=cats)
    model = CatBoostRegressor(**params)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True, verbose=False)
    return model, model.predict(val_pool)


_y_raw = pd.to_numeric(df_final[TARGET_COL], errors="coerce")
df_for_split = df_final.loc[_y_raw.notna()].reset_index(drop=True)

X_all, y_all, cat_cols_cb = build_feature_df(df_for_split)

if DATE_COL in df_for_split.columns:
    d_dt = pd.to_datetime(df_for_split[DATE_COL], errors="coerce")
    order = np.argsort(d_dt.fillna(d_dt.min()))
    cutoff = int((1.0 - 0.2) * len(df_for_split))
    tr_idx, va_idx = order[:cutoff], order[cutoff:]
else:
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    tr_idx, va_idx = next(kf.split(np.arange(len(df_for_split))))

X_tr, y_tr = X_all.iloc[tr_idx].reset_index(drop=True), y_all.iloc[tr_idx].reset_index(drop=True)
X_va, y_va = X_all.iloc[va_idx].reset_index(drop=True), y_all.iloc[va_idx].reset_index(drop=True)

y_tr_log = np.log(np.maximum(1e-6, y_tr))
y_va_log = np.log(np.maximum(1e-6, y_va))

brand_tr = df_for_split.iloc[tr_idx][BRAND_COL] if BRAND_COL in df_for_split.columns else pd.Series(["Unknown"] * len(tr_idx))
sample_weight = build_sample_weights(y_tr.values.astype(float), y_tr_log.values.astype(float), brand_tr)

print("Training CatBoost P10/P50/P90...")
m_p10, va_p10_log = train_quantile_model(X_tr, y_tr_log, X_va, y_va_log, 0.1, sample_weight)
m_p50, va_p50_log = train_quantile_model(X_tr, y_tr_log, X_va, y_va_log, 0.5, sample_weight)
m_p90, va_p90_log = train_quantile_model(X_tr, y_tr_log, X_va, y_va_log, 0.9, sample_weight)

va_p10, va_p50, va_p90 = np.exp(va_p10_log), np.exp(va_p50_log), np.exp(va_p90_log)

period_va = X_va["period"]
R_t = compute_residual_by_period(y_va_log.values, va_p50_log, period_va, agg="median")
gamma_t, M_t_raw = fit_time_dummy(R_t)
M_t_smooth = smooth_market_index(gamma_t, alpha=0.25)

M_vec = period_va.map(lambda x: M_t_smooth.get(x, M_t_raw.get(x, 1.0))).astype(float).values
va_p50_m = va_p50 * M_vec

brand_model_cal = compute_brand_model_calibration(y_va.values, va_p50_m, X_va)
brand_va = X_va[BRAND_COL].astype(str)
model_va = X_va[MODEL_COL].astype(str)
age_bin_va = X_va["car_age"].apply(age_bin)
coef_vec = np.array(
    [
        get_brand_model_multiplier(brand_va.iloc[i], model_va.iloc[i], age_bin_va.iloc[i], brand_model_cal)
        for i in range(len(y_va))
    ]
)

va_p10_mb = va_p10 * M_vec * coef_vec
va_p50_mb = va_p50 * M_vec * coef_vec
va_p90_mb = va_p90 * M_vec * coef_vec

group_keys = build_group_keys_for_cqr(X_va)
(q_lo_g, q_hi_g), (q_lo_groups, q_hi_groups) = cqr_asymmetric_ratio_global_and_group(
    y_va.values, va_p50_mb, va_p10_mb, va_p90_mb, group_keys, alpha=ALPHA, min_group_size=120
)

CATBOOST_MODELS = {"p10": m_p10, "p50": m_p50, "p90": m_p90}
CQR_META = {
    "columns": list(X_tr.columns),
    "categorical_cols": cat_cols_cb,
    "market_index": {"M_t_smooth": M_t_smooth, "M_t_raw": M_t_raw},
    "brand_model_calibration": brand_model_cal,
    "cqr_after_market": {"q_lo_global": q_lo_g, "q_hi_global": q_hi_g, "q_lo_groups": q_lo_groups, "q_hi_groups": q_hi_groups},
}
print("CatBoost Training and Meta Generation Complete.")

preds = va_p50_mb
targets = y_va.values

mae = np.mean(np.abs(preds - targets))
rmse = np.sqrt(np.mean((preds - targets) ** 2))

print("\nFinal Results on CatBoost Model (P50 Calibrated):")
print(f"MAE : {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")

# R^2 = 1 - (SS_res / SS_tot)
ss_total = np.sum((targets - np.mean(targets)) ** 2)
ss_res = np.sum((targets - preds) ** 2)
r2 = 1 - (ss_res / ss_total)
print(f"R^2 = {r2}")

Training CatBoost P10/P50/P90...
CatBoost Training and Meta Generation Complete.

Final Results on CatBoost Model (P50 Calibrated):
MAE : 258,787.33
RMSE: 661,741.29
R^2 = 0.9149378911795631


## Part 4: ResNet Model

In [4]:
class ResNetBlock(nn.Module):
    def __init__(self, input_dim, hidden_dim, dropout=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, input_dim),
            nn.BatchNorm1d(input_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.block(x)


class TabularResNet(nn.Module):
    def __init__(self, num_numerical, cat_dims, embedding_dims, hidden_dim=256, num_blocks=3, dropout=0.2):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(n, d) for n, d in zip(cat_dims, embedding_dims, strict=True)])
        self.num_bn = nn.BatchNorm1d(num_numerical)
        input_dim = sum(embedding_dims) + num_numerical
        self.input_proj = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU())
        self.resnet_layers = nn.ModuleList([ResNetBlock(hidden_dim, hidden_dim, dropout) for _ in range(num_blocks)])
        self.head = nn.Sequential(nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, x_num, x_cat):
        emb_outputs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x_emb = torch.cat(emb_outputs, dim=1)
        x_num = self.num_bn(x_num)
        x = torch.cat([x_emb, x_num], dim=1)
        x = self.input_proj(x)
        for layer in self.resnet_layers:
            x = layer(x)
        return self.head(x)


class CarDataset(data.Dataset):
    def __init__(self, X_num, X_cat, y_core, bias_brand, bias_fuel, raw_price):
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.y_core = torch.tensor(y_core, dtype=torch.float32)
        self.bias_brand = torch.tensor(bias_brand, dtype=torch.float32)
        self.bias_fuel = torch.tensor(bias_fuel, dtype=torch.float32)
        self.raw_price = torch.tensor(raw_price, dtype=torch.float32)

    def __len__(self):
        return len(self.y_core)

    def __getitem__(self, idx):
        return (
            self.X_num[idx],
            self.X_cat[idx],
            self.y_core[idx],
            self.bias_brand[idx],
            self.bias_fuel[idx],
            self.raw_price[idx],
        )


BATCH_SIZE = 64
LR = 2e-3
EPOCHS = RESNET_EPOCHS
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Preparing ResNet Data...")
df_res = df_final.copy()
df_res["Brand"] = df_res["Brand"].astype(str).str.strip()
df_res["Fuel Type"] = df_res["Fuel Type"].astype(str).str.strip()
df_res["Model"] = df_res["Model"].astype(str).str.strip()
df_res["Transmission"] = df_res["Transmission"].astype(str).str.strip()

impute_values = {}
for col in ["Engine", "Max Power"]:
    median_val = df_res[col].median()
    df_res[col] = df_res[col].fillna(median_val)
    impute_values[col] = median_val

df_res["log_price"] = np.log1p(df_res["Price"])
global_mean_log = df_res["log_price"].mean()

le_brand = LabelEncoder()
df_res["Brand_ID"] = le_brand.fit_transform(df_res["Brand"])
brand_map = (df_res.groupby("Brand_ID")["log_price"].mean() - global_mean_log).to_dict()
df_res["bias_brand"] = df_res["Brand_ID"].map(brand_map)

le_fuel = LabelEncoder()
df_res["Fuel_ID"] = le_fuel.fit_transform(df_res["Fuel Type"])
fuel_map = (df_res.groupby("Fuel_ID")["log_price"].mean() - global_mean_log).to_dict()
df_res["bias_fuel"] = df_res["Fuel_ID"].map(fuel_map)

df_res["y_core_raw"] = df_res["log_price"] - df_res["bias_brand"] - df_res["bias_fuel"]

le_model = LabelEncoder()
df_res["Model_ID"] = le_model.fit_transform(df_res["Model"])
le_trans = LabelEncoder()
df_res["Trans_ID"] = le_trans.fit_transform(df_res["Transmission"])

X_cat_res = df_res[["Model_ID", "Trans_ID"]].values
num_cols_res = ["Year", "Age", "Kilometer", "Engine", "Max Power", "Seats"]
X_num_res = df_res[num_cols_res].values

indices = np.arange(len(df_res))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)
train_core_values = df_res.iloc[train_idx]["y_core_raw"]
core_threshold = train_core_values.quantile(0.90)
clean_mask = train_core_values <= core_threshold
final_train_idx = train_idx[clean_mask]


def pack_data(idx_list):
    return (
        X_num_res[idx_list],
        X_cat_res[idx_list],
        df_res.iloc[idx_list]["y_core_raw"].values,
        df_res.iloc[idx_list]["bias_brand"].values,
        df_res.iloc[idx_list]["bias_fuel"].values,
        df_res.iloc[idx_list]["Price"].values,
    )


train_data = pack_data(final_train_idx)
test_data = pack_data(test_idx)

scaler_x = StandardScaler()
X_num_train = scaler_x.fit_transform(train_data[0])
X_num_test = scaler_x.transform(test_data[0])
scaler_y = StandardScaler()
y_core_train = scaler_y.fit_transform(train_data[2].reshape(-1, 1)).flatten()
y_core_test = scaler_y.transform(test_data[2].reshape(-1, 1)).flatten()

train_ds = CarDataset(X_num_train, train_data[1], y_core_train, train_data[3], train_data[4], train_data[5])
train_dl = data.DataLoader(train_ds, BATCH_SIZE, shuffle=True)
test_ds = CarDataset(X_num_test, test_data[1], y_core_test, test_data[3], test_data[4], test_data[5])
test_dl = data.DataLoader(test_ds, BATCH_SIZE, shuffle=False)

cat_dims = [len(le_model.classes_), len(le_trans.classes_)]
emb_dims = [min(50, (d + 1) // 2) for d in cat_dims]

RESNET_PREPROCESSOR = {
    "impute_values": impute_values,
    "global_mean_log": global_mean_log,
    "brand_map": brand_map,
    "fuel_map": fuel_map,
    "le_brand": le_brand,
    "le_fuel": le_fuel,
    "le_model": le_model,
    "le_trans": le_trans,
    "scaler_x": scaler_x,
    "scaler_y": scaler_y,
    "num_cols": num_cols_res,
}

RESNET_NET = TabularResNet(X_num_res.shape[1], cat_dims, emb_dims).to(DEVICE)
optimizer = torch.optim.AdamW(RESNET_NET.parameters(), lr=LR, weight_decay=1e-2)
criterion = nn.MSELoss()

print("Training ResNet Backbone...")
for epoch in range(EPOCHS):
    RESNET_NET.train()
    total_loss = 0
    for x_num, x_cat, y_core, _, _, _ in train_dl:
        x_num, x_cat, y_core = x_num.to(DEVICE), x_cat.to(DEVICE), y_core.to(DEVICE)
        optimizer.zero_grad()
        pred = RESNET_NET(x_num, x_cat).squeeze()
        loss = criterion(pred, y_core)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/{EPOCHS} | Train MSE: {total_loss / len(train_dl):.4f}")

RESNET_NET.eval()
results = []

with torch.no_grad():
    for x_num, x_cat, _, b_brand, b_fuel, raw_price in test_dl:
        x_num = x_num.to(DEVICE)
        x_cat = x_cat.to(DEVICE)
        
        pred_core_scaled = RESNET_NET(x_num, x_cat).squeeze()
        
        pred_core_scaled_cpu = pred_core_scaled.cpu().numpy().reshape(-1, 1)
        pred_core_raw = scaler_y.inverse_transform(pred_core_scaled_cpu).flatten()
        
        bias_brand_np = b_brand.numpy()
        bias_fuel_np = b_fuel.numpy()
        
        pred_log_price = pred_core_raw + bias_brand_np + bias_fuel_np
        
        pred_price = np.expm1(pred_log_price)
        
        target_price = raw_price.numpy()
        
        batch_res = np.vstack((pred_price, target_price)).T
        results.append(batch_res)

all_res = np.vstack(results)
preds = all_res[:, 0]
targets = all_res[:, 1]
mae = np.mean(np.abs(preds - targets))
rmse = np.sqrt(np.mean((preds - targets) ** 2))
print("\nFinal Results on Model:")
print(f"MAE : {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")

# R^2 = 1 - (SS_res / SS_tot)
ss_total = np.sum((targets - np.mean(targets)) ** 2)
ss_res = np.sum((targets - preds) ** 2)
r2 = 1 - (ss_res / ss_total)
print(f"R^2 = {r2}")

Preparing ResNet Data...
Training ResNet Backbone...
Epoch 10/100 | Train MSE: 0.1459
Epoch 20/100 | Train MSE: 0.1213
Epoch 30/100 | Train MSE: 0.1002
Epoch 40/100 | Train MSE: 0.0758
Epoch 50/100 | Train MSE: 0.0758
Epoch 60/100 | Train MSE: 0.0561
Epoch 70/100 | Train MSE: 0.0538
Epoch 80/100 | Train MSE: 0.0503
Epoch 90/100 | Train MSE: 0.0511
Epoch 100/100 | Train MSE: 0.0468

Final Results on Model:
MAE : 488,086.16
RMSE: 1,462,106.00
R^2 = 0.719581127166748


## Part 5: Meta Learning (Few-Shot Adapter)

In [5]:
@dataclass
class FewShotConfig:
    min_group_size: int = 30
    max_support_size: int = 200
    k_neighbors: int = 50
    new_car_max_age: float = 3.0
    iqr_to_width_factor: float = 1.8
    min_uncertainty_scale: float = 0.7
    max_uncertainty_scale: float = 1.6


class FewShotKnnMeta:
    def __init__(self, cfg: FewShotConfig, df_train):
        self.cfg = cfg
        self._df = df_train.copy()
        self._init_data()

    def _init_data(self):
        df = self._df
        column_map = {
            "Brand": "brand",
            "Model": "model",
            "Year": "year",
            "Age": "age",
            "Kilometer": "milage",
            "Fuel Type": "fuel_type",
            "Engine": "engine",
            "Max Power": "max_power",
            "Transmission": "transmission",
            "Seats": "seats",
            "Price": "price",
        }
        df = df.rename(columns=column_map)
        self._feat_cols = ["age", "milage", "engine", "max_power", "seats"]

        for col in ["year", "age", "milage", "engine", "max_power", "seats", "price"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        df["brand_key"] = df["brand"].astype(str).str.strip().str.lower()
        df["model_key"] = df["model"].astype(str).str.strip().str.lower()
        df["year_int"] = df["year"].astype("Int64").fillna(-1)
        df["_group_id"] = df["brand_key"] + "||" + df["model_key"] + "||" + df["year_int"].astype(str)

        self._group_counts = df["_group_id"].value_counts().to_dict()
        self._df = df

        X_raw = df[self._feat_cols].to_numpy(dtype=float)
        self._feature_means = np.nanmean(X_raw, axis=0)
        inds = np.where(np.isnan(X_raw))
        X_raw[inds] = np.take(self._feature_means, inds[1])
        self._feature_stds = np.nanstd(X_raw, axis=0)
        self._feature_stds[self._feature_stds == 0] = 1.0
        self._X = (X_raw - self._feature_means) / self._feature_stds
        self._y = df["price"].to_numpy(dtype=float)

    def maybe_adapt(self, d: dict[str, Any], base: dict[str, Any]) -> dict[str, Any]:
        cfg = self.cfg
        age_val = float(d.get("age") or 0)
        if age_val > cfg.new_car_max_age:
            return base

        brand_key = str(d.get("brand")).strip().lower()
        model_key = str(d.get("model")).strip().lower()
        year_int = int(d.get("year") or -1)
        group_id = f"{brand_key}||{model_key}||{year_int}"

        n_group = self._group_counts.get(group_id, 0)
        if n_group >= cfg.min_group_size:
            return base

        mask_family = (self._df["brand_key"] == brand_key) & (self._df["model_key"] == model_key)
        family_idx = np.where(mask_family)[0]
        if len(family_idx) == 0:
            return base

        if len(family_idx) > cfg.max_support_size:
            family_idx = family_idx[: cfg.max_support_size]

        query_vals = [d.get("age"), d.get("milage"), d.get("engine"), d.get("max_power"), d.get("seats")]
        x_query = np.array(query_vals, dtype=float)
        mask_nan = np.isnan(x_query)
        x_query[mask_nan] = self._feature_means[mask_nan]
        x_query_norm = (x_query - self._feature_means) / self._feature_stds

        X_family = self._X[family_idx]
        dist = np.sqrt(np.sum((X_family - x_query_norm) ** 2, axis=1))
        k = min(cfg.k_neighbors, len(dist))
        nearest = family_idx[np.argsort(dist)[:k]]
        y_knn = self._y[nearest]

        if len(y_knn) < 5:
            return base

        q50_local = np.percentile(y_knn, 50)
        p50_base = float(base.get("p50", 0))
        w = (1.0 - (n_group / cfg.min_group_size)) * (1.0 - (age_val / cfg.new_car_max_age))
        w = max(0.0, min(1.0, w))

        p50_meta = (1.0 - w) * p50_base + w * q50_local
        scale = p50_meta / p50_base if p50_base > 0 else 1.0

        out = base.copy()
        out["p50"] = p50_meta
        out["lo"] = base["lo"] * scale
        out["hi"] = base["hi"] * scale
        out["meta_info"] = {"enabled": True, "w": w}
        return out


FEWSHOT_ADAPTER = FewShotKnnMeta(FewShotConfig(), df_final)

# Inference Functions

In [7]:
def predict_catboost(d):
    _cols = CQR_META["columns"]
    _cat_cols = CQR_META["categorical_cols"]
    market_info = CQR_META["market_index"]
    bm_cal = CQR_META["brand_model_calibration"]
    cqr_info = CQR_META["cqr_after_market"]

    row = {}
    d_norm = {
        "Year": d.get("year"),
        "Age": d.get("age"),
        "Kilometer": d.get("milage"),
        "Max Power": d.get("max_power"),
        "Engine": d.get("engine"),
        "Seats": d.get("seats"),
    }

    for k, v in d_norm.items():
        val = float(v) if v is not None else np.nan
        row[k] = val if not np.isnan(val) else 0.0
        row[k + "_missing"] = 1 if np.isnan(val) else 0

    row["car_age"] = row["Age"]
    row["log1p_mileage"] = np.log1p(row["Kilometer"])
    age_eps = row["Age"] if row["Age"] > 0 else 0.25
    row["avg_km_per_year"] = row["Kilometer"] / age_eps

    fuel = str(d.get("fuel_type")).strip().lower()
    if fuel in ["petrol", "gasoline"]:
        f_std = "Petrol"
    elif fuel in ["diesel"]:
        f_std = "Diesel"
    else:
        f_std = "Other"

    gear = str(d.get("transmission")).strip().lower()
    if "auto" in gear:
        g_std = "Automatic"
    elif "man" in gear:
        g_std = "Manual"
    else:
        g_std = "Unknown"

    row["Brand"] = str(d.get("brand"))
    row["Model"] = str(d.get("model"))
    row["Fuel Type"] = f_std
    row["Transmission"] = g_std

    period = str(int(d.get("year"))) if d.get("year") else "Unknown"
    row["period"] = period
    y_val = int(d.get("year")) if d.get("year") else 0
    lo = y_val // 2 * 2
    row["period_bin"] = f"{lo}-{lo + 1}" if y_val > 0 else "Unknown"

    final_vec = []
    for c in _cols:
        if c in row:
            final_vec.append(row[c])
        else:
            final_vec.append("Unknown" if c in _cat_cols else 0.0)

    X = pd.DataFrame([final_vec], columns=_cols)

    p10 = np.exp(CATBOOST_MODELS["p10"].predict(X)[0])
    p50 = np.exp(CATBOOST_MODELS["p50"].predict(X)[0])
    p90 = np.exp(CATBOOST_MODELS["p90"].predict(X)[0])

    M = market_info["M_t_smooth"].get(period, 1.0)
    p50_m = p50 * M

    bm_coef = get_brand_model_multiplier(row["Brand"], row["Model"], age_bin(row["Age"]), bm_cal)
    p50_mb = p50_m * bm_coef
    p10_mb, p90_mb = p10 * M * bm_coef, p90 * M * bm_coef

    grp_key = f"{age_bin(row['Age'])}|{f_std}|{g_std}|{row['period_bin']}"
    q_lo = cqr_info["q_lo_groups"].get(grp_key, cqr_info["q_lo_global"])
    q_hi = cqr_info["q_hi_groups"].get(grp_key, cqr_info["q_hi_global"])

    lo_final = max(0, p10_mb - q_lo * p50_mb)
    hi_final = p90_mb + q_hi * p50_mb

    base_res = {"p50": p50_mb, "lo": lo_final, "hi": hi_final, "wr": (hi_final - lo_final) / p50_mb}
    return FEWSHOT_ADAPTER.maybe_adapt(d, base_res)


def predict_resnet(d):
    prep = RESNET_PREPROCESSOR

    row_data = {
        "Brand": str(d.get("brand")).strip(),
        "Model": str(d.get("model")).strip(),
        "Year": d.get("year"),
        "Age": d.get("age"),
        "Kilometer": d.get("milage"),
        "Fuel Type": str(d.get("fuel_type")).strip(),
        "Engine": d.get("engine"),
        "Max Power": d.get("max_power"),
        "Transmission": str(d.get("transmission")).strip(),
        "Seats": d.get("seats"),
    }

    df_in = pd.DataFrame([row_data])
    for c in ["Engine", "Max Power"]:
        df_in[c] = pd.to_numeric(df_in[c], errors="coerce").fillna(prep["impute_values"][c])

    X_num = prep["scaler_x"].transform(df_in[prep["num_cols"]].values)

    try:
        b_id = int(prep["le_brand"].transform(df_in["Brand"])[0])
    except (ValueError, KeyError, IndexError):
        b_id = -1
    b_bias = prep["brand_map"].get(b_id, 0.0)

    try:
        f_id = int(prep["le_fuel"].transform(df_in["Fuel Type"])[0])
    except (ValueError, KeyError, IndexError):
        f_id = -1
    f_bias = prep["fuel_map"].get(f_id, 0.0)

    try:
        m_id = int(prep["le_model"].transform(df_in["Model"])[0])
    except (ValueError, KeyError, IndexError):
        m_id = 0
    try:
        t_id = int(prep["le_trans"].transform(df_in["Transmission"])[0])
    except (ValueError, KeyError, IndexError):
        t_id = 0

    X_cat = np.array([[m_id, t_id]])

    RESNET_NET.eval()
    with torch.no_grad():
        t_num = torch.tensor(X_num, dtype=torch.float32).to(DEVICE)
        t_cat = torch.tensor(X_cat, dtype=torch.long).to(DEVICE)
        core_norm = RESNET_NET(t_num, t_cat).cpu().numpy().flatten()
        core_log = prep["scaler_y"].inverse_transform(core_norm.reshape(-1, 1)).flatten()[0]

    final_log = core_log + b_bias + f_bias
    return float(np.expm1(final_log))


payload = {
    "brand": "Toyota",
    "model": "Corolla",
    "year": 2019,
    "age": 6,
    "milage": 45000,
    "fuel_type": "Petrol",
    "engine": 1800,
    "max_power": 140,
    "transmission": "Automatic",
    "seats": 5,
}

cb_res = predict_catboost(payload)
rn_price = predict_resnet(payload)

print("Prediction:")
print(f"  CatBoost: {cb_res['p50'] / INR2USD:,.2f} (Interval: {cb_res['lo'] / INR2USD:,.2f} - {cb_res['hi'] / INR2USD:,.2f})")
print(f"  ResNet:   {rn_price / INR2USD:,.2f}")

Prediction:
  CatBoost: 15,021.74 (Interval: 9,352.55 - 14,516.02)
  ResNet:   24,114.15
